# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 5: *Feature Interaction Analysis*
##### Version Number: 2.0
---
### Contents  
> 1. *Filter Dates for modeling*
> 2. *Split and Scale Data*
> 3. *Export Data*
---
### Notes
---
### Inputs
- `final.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `X`,`y` - for future modeling
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions
from src.data_utils import create_interactions

# A space saving function to rank interactions
from src.data_utils import rank_interactions_by_correlation

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling prep
from sklearn.preprocessing import MinMaxScaler

---

###  Load Data

In [3]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/filtered.csv")

## 1. Filter Dates for modeling

In [4]:
final['Date'] = pd.to_datetime(final['Date'])

# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

In [5]:
# Boolean mask for dates within the range
mask = (final['Date'].dt.date >= FIRST_DATE) & (final['Date'].dt.date <= LAST_DATE)

pal = final.loc[~mask]

# Keep only rows within the range
final = final.loc[mask].copy()

In [6]:
pal['Date'].min()

Timestamp('2025-01-01 00:00:00')

In [7]:
final['Date'].max()

Timestamp('2024-12-31 00:00:00')

In [8]:
final['Date'] = pd.to_datetime(final['Date'])

# Number of rows to keep from class 0
n_keep = 50000

# Split by class
class0 = final[final['Target'] == 0]
class1 = final[final['Target'] == 1]
class2 = final[final['Target'] == 2]

# Sample class 0 down to 50,000 rows
class0_sampled = class0.sample(n=n_keep, random_state=14)

# Combine back together
reduced = pd.concat([class0_sampled, class1, class2], ignore_index=True)

In [9]:
reduced['Target'].value_counts()

Target
0    50000
1    43005
2    13773
Name: count, dtype: int64

## 2. Split and Scale Data

In [10]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target', 
      'Region_Name','Eco_Name',
       'County', 'Sample_Longitude', 'Sample_Latitude',
       'geometry', 'fire_count', 'total_fire_damage',
        '_merge',]

y = reduced['Target']
X = reduced.drop(columns=text_columns)
details = reduced[text_columns]

pal_details = pal[text_columns]
pal_X = pal.drop(columns=text_columns)

Scale all datasets and save back to dataframe format. MinMax Scaler used because majority of values are > 0

In [11]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

In [12]:
X_scaled

,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,Specific Humidity,Solar Radiation,...,Year,Vapor Pressure Deficit 7 Day Avg,Precipitation 7 Day Avg,Solar Radiation 7 Day Avg,Daily Maximum Air Temperature 7 Day Avg,Daily Minimum Air Temperature 7 Day Avg,Maximum Relative Humidity 7 Day Avg,Minimum Relative Humidity 7 Day Avg,Wind Speed 7 Day Avg,2-Year Avg Fires
0,0.463576,0.586777,0.373950,0.159091,0.222535,0.0,0.437302,0.098990,0.182482,0.624699,...,0.000000,0.185892,0.000000,0.744075,0.524180,0.599906,0.455418,0.145028,0.309406,0.028516
1,0.251656,0.314050,0.210084,0.448052,0.357746,0.0,1.000000,0.381818,0.322211,0.762925,...,0.000000,0.111150,0.000000,0.814105,0.526367,0.519603,0.947004,0.324125,0.299505,0.030136
2,0.000000,0.173554,0.117647,0.509740,0.411268,0.2,0.700738,0.473737,0.145985,0.570051,...,0.333333,0.047961,0.076724,0.455645,0.310328,0.441427,0.802485,0.487569,0.122525,0.118276
3,0.198675,0.305785,0.159664,0.386364,0.428169,0.0,0.524763,0.100000,0.163191,0.259041,...,1.000000,0.057413,0.020657,0.176226,0.399271,0.511809,0.832364,0.456875,0.205446,0.660726
4,0.357616,0.570248,0.252101,0.275974,0.185915,0.0,0.629083,0.133333,0.200209,0.375837,...,0.500000,0.187467,0.000000,0.351506,0.591251,0.555503,0.699481,0.198128,0.298267,0.553467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106773,0.423841,0.669421,0.369748,0.172078,0.140845,0.0,0.476291,0.144444,0.290407,0.758103,...,1.000000,0.236478,0.003689,0.733994,0.635966,0.650213,0.558421,0.193524,0.196782,1.000000
106774,0.390728,0.611570,0.264706,0.240260,0.143662,0.0,0.782929,0.310101,0.338895,0.736137,...,1.000000,0.192018,0.000000,0.734036,0.571567,0.608644,0.566599,0.213475,0.217822,1.000000
106775,0.344371,0.570248,0.310924,0.243506,0.205634,0.0,0.440464,0.088889,0.181439,0.690329,...,1.000000,0.124978,0.018443,0.615495,0.468287,0.528106,0.691933,0.323818,0.205446,1.000000
106776,0.344371,0.545455,0.235294,0.198052,0.233803,0.0,0.496312,0.130303,0.165276,0.505759,...,1.000000,0.181516,0.000000,0.505802,0.549453,0.568021,0.533103,0.170810,0.240099,0.947829


---

## 3. Export Data

In [15]:
X.to_csv('../data/processed/X.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)
details.to_csv('../data/processed/details.csv', index=False)
pal_X.to_csv('../data/processed/pal_X.csv', index=False)
pal_details.to_csv('../data/processed/pal_details.csv', index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
